In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv
/kaggle/input/competitions/comment-category-prediction-challenge/train.csv
/kaggle/input/competitions/comment-category-prediction-challenge/test.csv


# 7. MODEL TRAINING

## 7.1 Logistic Regression (class-weighted)

In [14]:
lr = LogisticRegression(max_iter=1000, C=1.5, solver="lbfgs",
                         multi_class="multinomial", class_weight="balanced", n_jobs=-1)
lr.fit(Xs_tr, y_tr)
val_probas["lr"] = lr.predict_proba(Xs_val)
acc_scores["lr"], f1_scores["lr"] = evaluate("Logistic Regression", val_probas["lr"])
print("elapsed:", elapsed(), "s")


Logistic Regression      | val accuracy = 0.8886 | macro-F1 = 0.7728
elapsed: 82.2 s


## 7.2 Complement Naive Bayes

In [15]:
cnb = ComplementNB(alpha=0.3)
cnb.fit(Xw_tr, y_tr)
val_probas["cnb"] = cnb.predict_proba(Xw_val)
acc_scores["cnb"], f1_scores["cnb"] = evaluate("Complement NB", val_probas["cnb"])
print("elapsed:", elapsed(), "s")


Complement NB            | val accuracy = 0.6924 | macro-F1 = 0.6114
elapsed: 82.2 s


## 7.3 SGD (log-loss, class-weighted)

In [16]:
sgd = SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=80, class_weight="balanced",
                     random_state=RANDOM_STATE, n_jobs=-1)
sgd.fit(Xs_tr, y_tr)
val_probas["sgd"] = sgd.predict_proba(Xs_val)
acc_scores["sgd"], f1_scores["sgd"] = evaluate("SGD (log-loss)", val_probas["sgd"])
print("elapsed:", elapsed(), "s")


SGD (log-loss)           | val accuracy = 0.8874 | macro-F1 = 0.7754
elapsed: 86.9 s


## 7.4 LightGBM

In [17]:
lgbm = LGBMClassifier(
    n_estimators=150, learning_rate=0.09, num_leaves=31, feature_fraction=0.3,
    subsample=0.8, max_bin=127, min_child_samples=30, class_weight="balanced",
    objective="multiclass", num_class=n_classes, random_state=RANDOM_STATE,
    n_jobs=-1, verbosity=-1,
)
lgbm.fit(Xs_tr, y_tr)
val_probas["lgbm"] = lgbm.predict_proba(Xs_val)
acc_scores["lgbm"], f1_scores["lgbm"] = evaluate("LightGBM", val_probas["lgbm"])
print("elapsed:", elapsed(), "s")


LightGBM                 | val accuracy = 0.8919 | macro-F1 = 0.7732
elapsed: 192.4 s


## 7.5 Random Forest (numeric/engineered features only)

In [18]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced_subsample",
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(Xn_tr, y_tr)
val_probas["rf"] = rf.predict_proba(Xn_val)
acc_scores["rf"], f1_scores["rf"] = evaluate("Random Forest (numeric)", val_probas["rf"])
print("elapsed:", elapsed(), "s")


Random Forest (numeric)  | val accuracy = 0.7854 | macro-F1 = 0.5896
elapsed: 211.5 s
